In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-5: PPO in depth: credit for every token

**RL for LLMs, session 5.**


## What this session teaches

A model writes an answer one token at a time, but the reward arrives once, at the end.

Each token needs its own **weight**, meaning a number that says push it up or down. The hard question is which tokens earned the reward.

PPO answers with a second network. The **critic** is a network that predicts, for the text so far, the reward still to come.

The answer's **total surprise** is its reward minus what the critic first expected.

Credit assignment splits that total across the tokens.

Each new token changes the prediction. A token's surprise is how much better the outcome looks after it than the critic expected just before: the reward it earned plus the change in the prediction.

The token surprises add up to the total.

In the simplest case, a token's surprise is its credit: positive, it helped; negative, it hurt.

In practice PPO also gives each token a share of the later surprises; we add that once the simple case is clear.

The price is a second network that must keep up with the learner.

The question to carry: what does credit for each token buy, and what is lost without it?

By the end you can

- say how the critic gives each token a credit
- say what a critic costs
- say what plain GRPO gives up by dropping it

**Where this session fits: the topics of the series, in order**

<img src="images/rllm5v2_strip.png" width=88%>

- this session is the first of topic 5: a reward becomes a weight for each token (PPO now, GRPO's refinements in RLLM-6)


## Setting and words

**Why this slide?** The credit slides use words that the opening only previews; each is said by what it does.

| word | what it means here |
|:--|:--|
| **prefix** | the text written so far; the critic reads it, and the policy continues it (RLLM-2's token view) |
| **return** | the total reward from a token to the end of the completion; the total the critic predicts |
| **TD error** | the critic's surprise at one token: the reward the token earned plus the change in the critic's prediction (Intro B3) |
| **advantage** (also **credit**, and the **weight**) | the number that pushes a token up (positive) or down (negative); the Intro's word is advantage, this session says credit for a token's own |
| **lambda** | the setting that decides how much of the later surprises a token receives: $0$ means none (the simplest case), near $1$ means almost all |
| **discount** | set to 1 in this session, so nothing is discounted and later reward counts in full |


## The map: where credit comes from

**Why this slide?** One reward at the end must become a weight for every token. These five sections show how PPO does it.

<img src="images/rllm5v2_map.png" width=96%>

<img src="images/rllm5v2_onereward.png" width=48%>

- RLLM-2's token view: the **state** is the prefix, the **action** is the next token
- the reward model's reward lands on the last token, so the early tokens see none


## Four networks and one loop

<img src="images/rllm5v2_bar_1.png" width=70%>

**You are here:** section 1 of 5

**Why this section?** The first confusion on a language model is how many networks there are, and which ones learn.

> **Four networks matter. The policy and the critic are updated; the reference model and the reward model are frozen.**

| network | input and output | updated? | job |
|:--|:--|:--|:--|
| **policy** | prefix to next-token probabilities | yes | writes the completions |
| **critic** | prefix to its value | yes | predicts the return from here |
| **reference model** | prefix to next-token probabilities | no | the SFT model: the anchor of the KL reward (RLLM-2) |
| **reward model** | prompt and completion to a reward | no | the reward at the last token (RLLM-2) |

**One iteration of a PPO run: an illustration, not the sections**

<img src="images/rllm5v2_loop.png" width=74%>

- the third box is this session's subject: the critic turns rewards into one advantage per token
- the reward at each token is a small KL term, and the reward model's reward is added at the last token (Intro C1)
- the batch is reused for several steps, so it goes stale (Safe update)


## Credit: a weight for every token

<img src="images/rllm5v2_bar_2.png" width=70%>

**You are here:** section 2 of 5

**Why this section?** This is the point of the session: the policy needs a weight per token, and the critic is where it comes from.

> **Credit assignment: decide which tokens earned the reward. A token's credit is its advantage, the weight on its push.**

- the critic has **three jobs**: baseline and bootstrap, which the Intro taught (B3), and credit assignment, which this session is about
- **baseline:** subtracting the critic's value from the return lowers the noise in the weights
- **bootstrap:** the critic's value stands in for the rest of the return
- **credit assignment:** a value exists for every prefix, so every token can get its own weight
- plain GRPO sacrifices credit assignment (the last section)


## The critic values every prefix

**Why this slide?** A weight per token needs a number per token. The value of each prefix is where it comes from.

> **The value is the critic's prediction of the return from this prefix. Across one completion it moves, token by token.**

<img src="images/rllm5v2_value.png" width=64%>

- the prefix is the **state** of RLLM-2's token view
- the critic reads it and predicts the return still to come
- this is an estimate, only as good as the critic (**our interpretation**)
- the value after the last token is $0$: nothing is left to come
- the path is made up, to show the shape: it rises, flattens and drops


## The critic's surprise is the token's credit

**Why this slide?** Credit must be local: a token that made the outcome look better helped, one that made it look worse hurt.

> **The TD error is the critic's surprise: the reward the token earned plus the move in the value. In the simplest case (lambda = 0) it is the token's credit.**

<img src="images/rllm5v2_value_td.png" width=60%>

- the formula: $\delta_{\tt} = \rew_{\tt} + \statevalfun_\phi(\text{next prefix}) - \statevalfun_\phi(\text{prefix})$ with no discounting (Intro B3)
- token 1 earned $-0.10$, yet the value rose by $0.20$, so its surprise is $+0.10$: it helped
- token 4 ended the completion and the value fell to $0$: a surprise of $-0.35$, so it hurt
- in practice a token also gets a share of later surprises (lambda, below)


## The surprises add up

**Why this slide?** Why is a jump in the value a fair credit? Because credit splits the total; nothing is lost or invented.

> **The surprises of all tokens add up to the answer's total surprise: its reward minus the critic's first value.**

- with no discounting the value terms cancel in pairs, and the value after the last token is $0$
- so the four surprises add up to the total reward minus the critic's first value
- surprises: $0.10 + 0.00 + (-0.10) + (-0.35) = -0.35$
- total reward $-0.05$ (the four rewards added) minus the critic's first value $0.30$ is $-0.35$: the same
- this holds for the surprises themselves, whatever lambda is


## The same surprise has three jobs

**Why this slide?** The student met the surprise in the Intro as a noise cutter. Credit is not a different quantity.

> **The TD error, the critic's surprise, does three jobs. Baseline and bootstrap cut noise, as in the Intro; credit assignment tells tokens apart.**
<!-- seed-plant: critic-two-roles -->

| job | what it gives | what it costs |
|:--|:--|:--|
| **baseline** | a weight that is above or below what the critic expected, not above zero | a critic that must be trained |
| **bootstrap** | less noise: the value replaces most of the return that has not yet happened | bias when the critic is wrong |
| **credit assignment** | a weight for every token | the same critic, and its errors reach each token |

- the Intro called the three jobs roles (B3)
- a per-token weight also exists without a critic: the sampled return to the end, minus a baseline
- the critic is what makes that weight low in noise


## In practice, a share of the later surprises

**Why this slide?** The opening promised this refinement. Without it the simple rule disagrees with every real PPO run.

> **In practice (lambda about 0.95) a token gets its own surprise plus a decaying share of the later ones. Lambda sets the share.**

<img src="images/rllm5v2_lambda.png" width=46%>

- lambda $= 0$: own surprise only, the simplest case
    - token 1 is praised ($+0.10$)
- lambda $= 1$: all later surprise counts; token 1 gets $-0.35$, the whole shortfall
- lambda about $0.95$: token 1 gets $-0.29$, blamed through the later errors it leads to
- the recursion is Intro B3's: $\hat{\advseq}_{\tt} = \delta_{\tt} + \gamma \lambda\, \hat{\advseq}_{\tt+1}$
- small lambda: less noise, more bias from the critic
- large lambda: the reverse (**our interpretation**)


## How much does per-token credit buy?

<img src="images/rllm5v2_bar_2.png" width=70%>

**You are here:** section 2 of 5, its last slide

**Why this slide?** Per-token credit is the reason to pay for a critic. The reports on whether it pays are mixed.

> **In principle per-token credit is a large gain. In practice it is only as good as the critic, and on long answers a good critic took extra work.**

- **for:** VAPO, a PPO with a critic, reports 60.4 on AIME 2024 with a Qwen 32B base ([arXiv 2504.05118](https://arxiv.org/abs/2504.05118))
    - more than 10 points above DeepSeek-R1-Zero-Qwen-32B and DAPO, under identical settings (**fact**, the authors' report)
    - it names three obstacles: critic bias, answers of very different lengths, and sparse reward
- **a warning:** VC-PPO finds that standard PPO fails on long reasoning chains ([arXiv 2503.01491](https://arxiv.org/abs/2503.01491))
    - two causes: the critic's starting bias, and reward signal that decays along the answer (**fact**, the authors' report)
- **against:** "Back to Basics" finds simpler REINFORCE-style methods beat PPO in RLHF ([arXiv 2402.14740](https://arxiv.org/abs/2402.14740)) (**fact**, the authors' report)
- our interpretation: credit pays only when the critic is accurate, and that has needed dedicated fixes on long answers
- each report is one team's, on its own base model and benchmark
- none settles the question


## How the critic is trained

<img src="images/rllm5v2_bar_3.png" width=70%>

**You are here:** section 3 of 5

**Why this section?** Credit is only as good as the critic, so we ask what it is trained to match.

> **The critic is trained by regression to a target built from this batch: the token's advantage plus the critic's own earlier value.**

<img src="images/rllm5v2_target.png" width=34%>

- token 1: advantage $-0.29$ plus the earlier value $0.30$ gives a target of $0.01$, so its value is pulled from $0.30$ toward it
- the value loss is the mean squared gap between the critic's value and the target
- the critic is often the policy's network with a new output layer, trained on the same mini-batches (common in practice, **our interpretation**)
- with lambda $= 1$ the target is exactly the return


## What the critic costs, and where it fails

**Why this slide?** The price of credit is the reason the next sessions look elsewhere.

> **The critic is a second network, starting with meaningless values, that must keep up with a policy that keeps moving.**

- **memory and compute:** its own parameters, gradients and optimizer state
    - a critic of the policy's size roughly doubles what is trained (a rough figure, **our interpretation**)
- **the cold start:** the critic's output layer begins random, so early weights are noisy
- **a critic that falls behind:** the policy changes, and the critic's targets change with it
    - a lagging critic gives noisy weights, and the policy follows noise (**our interpretation**)
- **long completions:** one end reward is spread over hundreds of tokens (**our interpretation**)
    - VC-PPO and VAPO study this failure and its fixes (the slide on what credit buys)


## The clip, a brake on a stale batch

<img src="images/rllm5v2_bar_4.png" width=70%>

**You are here:** section 4 of 5

**Why this section?** The weights are applied for several steps on one batch, and after the first step the batch is stale.

> **PPO stops pushing a token once its probability has moved more than a small amount from the old policy's. That brake does not bound the KL.**
<!-- seed-plant: clip-heuristic -->

<img src="images/rllm4_clip.png" width=42%>

- the **old policy** wrote the batch
- the **ratio** is a token's probability now over its probability then (Intro B4)
- the clip zeroes the push once the ratio passes $1 \pm \epsilon_{\text{clip}}$, typically $\epsilon_{\text{clip}} = 0.2$, in the direction the advantage asks
- with one step per batch the ratio is $1$ and the clip never acts
- **two brakes:** the clip limits drift within one batch
    - the KL reward limits drift across the run (RLLM-2)
- our interpretation: the clip gives no guarantee on the KL; RLLM-6 returns to this


## PPO versus GRPO, piece by piece

<img src="images/rllm5v2_bar_5.png" width=70%>

**You are here:** section 5 of 5

**Why this section?** This is the trade the session was building to: per-token credit has a price, and GRPO chooses not to pay it.

> **PPO buys per-token credit with a critic. Plain GRPO sacrifices it to remove the critic.**

| piece of an update | PPO (this session) | plain GRPO |
|:--|:--|:--|
| weight | one per token, from the critic | one per answer, from the group |
| baseline | the critic's value | the group's mean reward |
| clip | per token, the same clip | per token, the same clip |
| networks trained | two: policy and critic | one: the policy |

- GRPO in one line (RLLM-4): sample a group of answers, score each against the group's mean and spread
    - every token of an answer is pushed by that one weight


## What the critic's credit shows in the example

**Why this slide?** The same four tokens, under both methods, show what is lost.

> **In the worked example PPO tells the four tokens apart; plain GRPO cannot.**

<img src="images/rllm4_credit.png" width=60%>

- PPO: four weights, shown for lambda about $0.95$, each from the critic's surprises
- GRPO: one weight for the whole answer, computed from the group's rewards, so no per-token bars
- GRPO compensates by sampling many answers per question instead of training a critic
- the next sessions meet methods that refine GRPO's one weight per answer (RLLM-6)


# What we established, and what it sets up

- **Established**
    - the policy needs a weight for every token
    - in the simplest case the critic's surprise at a token is its credit
    - the surprises add up to the answer's total surprise
    - in practice lambda gives each token a share of the later ones
    - the critic has three jobs: baseline, bootstrap and credit assignment
    - per-token credit is a large gain only when the critic is accurate, and the reports are mixed
    - its price is a second network that must keep up
    - the clip is a heuristic brake on a stale batch and does not bound the KL
    - plain GRPO sacrifices per-token credit in order to remove the critic
- **Sets up**
    - RLLM-6: GRPO and the refinements that later papers report, including credit assignment as an open problem


## Aside: the critic's target, token by token

| | token 1 | token 2 | token 3 | token 4 |
|:--|:--|:--|:--|:--|
| advantage, lambda $= 0.95$ | -0.29 | -0.41 | -0.43 | -0.35 |
| critic's value before the update | 0.30 | 0.50 | 0.45 | 0.45 |
| **target** = advantage + value | +0.01 | +0.09 | +0.02 | +0.10 |


## Aside: where the clip comes from, TRPO

> **TRPO bounds the KL between old and new policy at every step. PPO's clip is a cheaper stand-in with no such bound.**

- **TRPO** (Schulman et al. 2015) maximizes the surrogate objective subject to a bound on the KL between old and new policy
- PPO's clip limits only the ratio of the sampled token: our interpretation, no bound on the KL, so a heuristic
